In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import json
import warnings

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.ensemble import (
    RandomForestRegressor,
    RandomForestClassifier
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [2]:
thermal = pd.read_csv("final_thermal_dataset.csv")

soh_rul = pd.read_csv("final_soh_rul_dataset.csv")

health = pd.read_csv("final_health_dataset.csv")

degradation = pd.read_csv("final_degradation_dataset.csv")

cycle_life = pd.read_csv("final_cycle_life_dataset.csv")

feature_store = pd.read_csv("battery_asset_feature_store.csv")


Before merging anything, we need to understand:

how many rows each dataset contains;
which columns are available;
what identifies a battery;
what identifies a cycle;
which columns are features;
which columns are targets.

In [3]:
datasets = {
    "Thermal": thermal,
    "SOH_RUL": soh_rul,
    "Health": health,
    "Degradation": degradation,
    "Cycle_Life": cycle_life,
    "Feature_Store": feature_store
}

for name, df in datasets.items():

    print("\n", "=" * 50)
    print(name)
    print("=" * 50)

    print("Shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())

    print("\nFirst 5 rows:")
    display(df.head())


Thermal
Shape: (71, 17)

Columns:
['Trip', 'Date', 'Route/Area', 'Weather', 'Battery Temperature (Start) [°C]', 'Battery Temperature (End)', 'Battery State of Charge (Start)', 'Battery State of Charge (End)', 'Unnamed: 8', 'Ambient Temperature (Start) [°C]', 'Target Cabin Temperature', 'Distance [km]', 'Duration [min]', 'Unnamed: 13', 'Fan', 'Note', 'Average_Speed']

First 5 rows:


,Trip,Date,Route/Area,Weather,Battery Temperature (Start) [°C],Battery Temperature (End),Battery State of Charge (Start),Battery State of Charge (End),Unnamed: 8,Ambient Temperature (Start) [°C],Target Cabin Temperature,Distance [km],Duration [min],Unnamed: 13,Fan,Note,Average_Speed
0,TripA01,2019-06-25_13-21-14,Munich East,sunny,21.0,22.0,0.863,0.803,0.060,25.5,23.0,7.427690,16.820000,NaN,"Automatic, Level 1",NaN,0.441599
1,TripA02,2019-06-25_14-05-31,Munich East,sunny,23.0,26.0,0.803,0.673,0.130,32.0,23.0,23.509709,23.550000,NaN,"Automatic, Level 1",Target Cabin Temperature changed,0.998289
2,TripA03,2019-06-28_10-02-15,Munich East,sunny,24.0,25.0,0.835,0.751,0.084,21.5,27.0,12.820846,11.180000,NaN,"Automatic, Level 1",Target Cabin Temperature changed,1.146766
3,TripA04,2019-06-28_10-13-30,Munich East,sunny,25.0,27.0,0.751,0.667,0.084,24.0,22.0,10.727491,6.870000,NaN,"Automatic, Level 1",NaN,1.561498
4,TripA05,2019-06-28_10-20-26,Munich East,sunny,27.0,27.0,0.667,0.602,0.065,24.5,24.0,12.393223,22.776667,NaN,"Automatic, Level 1",NaN,0.544119



SOH_RUL
Shape: (1415, 12)

Columns:
['battery_id', 'cycle', 'voltage', 'temperature', 'capacity', 'soh', 'rul', 'capacity_loss', 'capacity_retention', 'voltage_drop_rate', 'temperature_change_rate', 'cycle_percentage']

First 5 rows:


,battery_id,cycle,voltage,temperature,capacity,soh,rul,capacity_loss,capacity_retention,voltage_drop_rate,temperature_change_rate,cycle_percentage
0,B0005,1,3.532781,32.536891,1.861976,1.000000,167,0.000173,99.990702,0.000000,0.000000,0.595238
1,B0005,2,3.542968,32.643595,1.851862,0.994568,166,0.010288,99.447542,0.010187,0.106704,1.190476
2,B0005,3,3.553056,32.522526,1.840808,0.988631,165,0.021342,98.853917,0.010088,-0.121069,1.785714
3,B0005,4,3.545849,32.492083,1.850058,0.993599,164,0.012091,99.350673,-0.007208,-0.030443,2.380952
4,B0005,5,3.544456,32.368612,1.849432,0.993263,163,0.012717,99.317073,-0.001392,-0.123471,2.976190



Health
Shape: (7368, 13)

Columns:
['type', 'ambient_temperature', 'battery_id', 'test_id', 'uid', 'filename', 'Capacity', 'Re', 'Rct', 'Resistance_Ratio', 'Normalized_Capacity', 'SOH_Estimate', 'Health_Class']

First 5 rows:


...

,Data_Point,Test_Time(s),Date_Time,Step_Time(s),Step_Index,Cycle_Index,Current(A),Voltage(V),Charge_Capacity(Ah),Discharge_Capacity(Ah),Charge_Energy(Wh),Discharge_Energy(Wh),dV/dt(V/s),Internal_Resistance(Ohm),Is_FC_Data,AC_Impedance(Ohm),ACI_Phase_Angle(Deg)
0,1,1.001205,2015-10-16 09:34:19,1.001206,1,1,0.0,3.434257,0.0,0.0,0.0,0.0,0.000000,0.0,0,0,0
1,2,2.016763,2015-10-16 09:34:20,2.016764,1,1,0.0,3.434418,0.0,0.0,0.0,0.0,0.000032,0.0,0,0,0
2,3,3.032254,2015-10-16 09:34:21,3.032255,1,1,0.0,3.434581,0.0,0.0,0.0,0.0,0.000065,0.0,0,0,0
3,4,4.032554,2015-10-16 09:34:22,4.032555,1,1,0.0,3.434581,0.0,0.0,0.0,0.0,0.000065,0.0,0,0,0
4,5,5.047983,2015-10-16 09:34:23,5.047983,1,1,0.0,3.434257,0.0,0.0,0.0,0.0,0.000000,0.0,0,0,0



Cycle_Life
Shape: (7000, 17)

Columns:
['IR', 'QC', 'QD', 'Tavg', 'Tmin', 'Tmax', 'chargetime', 'cycle', 'battery_id', 'cycle_life', 'C1', 'Q1', 'C2', 'Temperature_Range', 'Normalized_IR', 'Cycle_Percentage', 'Charge_Discharge_Ratio']

First 5 rows:


,IR,QC,QD,Tavg,Tmin,Tmax,chargetime,cycle,battery_id,cycle_life,C1,Q1,C2,Temperature_Range,Normalized_IR,Cycle_Percentage,Charge_Discharge_Ratio
0,0.016742,1.071042,1.070689,31.875011,29.566130,35.652016,13.341250,2,b1c0,1190.0,3.6,80.0,3.6,6.085886,0.336067,2.469136,0.999670
1,0.016724,1.071674,1.071900,31.931490,29.604385,35.692978,13.425777,3,b1c0,1190.0,3.6,80.0,3.6,6.088593,0.318293,3.703704,1.000211
2,0.016681,1.072304,1.072510,31.932603,29.744202,35.680588,13.425167,4,b1c0,1190.0,3.6,80.0,3.6,5.936386,0.275876,4.938272,1.000191
3,0.016662,1.072970,1.073174,31.959322,29.644709,35.728691,13.341442,5,b1c0,1190.0,3.6,80.0,3.6,6.083982,0.256789,6.172840,1.000190
4,0.016623,1.073491,1.073576,31.961062,29.752932,35.711758,13.340835,6,b1c0,1190.0,3.6,80.0,3.6,5.958826,0.218883,7.407407,1.000079



Feature_Store
Shape: (1415, 5)

Columns:
['battery_id', 'cycle', 'soh', 'rul', 'capacity_retention']

First 5 rows:


,battery_id,cycle,soh,rul,capacity_retention
0,B0005,1,1.000000,167,99.990702
1,B0005,2,0.994568,166,99.447542
2,B0005,3,0.988631,165,98.853917
3,B0005,4,0.993599,164,99.350673
4,B0005,5,0.993263,163,99.317073


**Create the SOH and RUL master table**

In [4]:
soh_master = feature_store.merge(
    soh_rul,
    on=["battery_id", "cycle"],
    how="inner",
    suffixes=("_store", ""),
    validate="one_to_one"
)


#for result checking

print("Feature Store Shape:")
print(feature_store.shape)

print("\nSOH/RUL Shape:")
print(soh_rul.shape)

print("\nMerged Shape:")
print(soh_master.shape)

display(soh_master.head())

Feature Store Shape:
(1415, 5)

SOH/RUL Shape:
(1415, 12)

Merged Shape:
(1415, 15)


,battery_id,cycle,soh_store,rul_store,capacity_retention_store,voltage,temperature,capacity,soh,rul,capacity_loss,capacity_retention,voltage_drop_rate,temperature_change_rate,cycle_percentage
0,B0005,1,1.000000,167,99.990702,3.532781,32.536891,1.861976,1.000000,167,0.000173,99.990702,0.000000,0.000000,0.595238
1,B0005,2,0.994568,166,99.447542,3.542968,32.643595,1.851862,0.994568,166,0.010288,99.447542,0.010187,0.106704,1.190476
2,B0005,3,0.988631,165,98.853917,3.553056,32.522526,1.840808,0.988631,165,0.021342,98.853917,0.010088,-0.121069,1.785714
3,B0005,4,0.993599,164,99.350673,3.545849,32.492083,1.850058,0.993599,164,0.012091,99.350673,-0.007208,-0.030443,2.380952
4,B0005,5,0.993263,163,99.317073,3.544456,32.368612,1.849432,0.993263,163,0.012717,99.317073,-0.001392,-0.123471,2.976190


Remove duplicate target columns

In [5]:
duplicate_columns = [
    "soh_store",
    "rul_store",
    "capacity_retention_store"
]

for column in duplicate_columns:

    if column in soh_master.columns:

        soh_master.drop(
            columns=column,
            inplace=True
        )


In [6]:
print(soh_master.columns.tolist())

['battery_id', 'cycle', 'voltage', 'temperature', 'capacity', 'soh', 'rul', 'capacity_loss', 'capacity_retention', 'voltage_drop_rate', 'temperature_change_rate', 'cycle_percentage']


**FEATURE SELECTION**

In [7]:
SOH_FEATURES = [
    "cycle",
    "voltage",
    "temperature",
    "capacity",
    "capacity_loss",
    "capacity_retention",
    "voltage_drop_rate",
    "temperature_change_rate",
    "cycle_percentage"
]

#target variable

SOH_TARGET = "soh"


X_soh = soh_master[SOH_FEATURES] #X → Input features

y_soh = soh_master[SOH_TARGET]  #y → Target

Now splitting the adata (not random as same battery can have different cycles)

In [8]:
def group_split(
    X,
    y,
    groups,
    test_size=0.20
):

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=42
    )

    train_index, test_index = next(
        splitter.split(
            X,
            y,
            groups=groups
        )
    )

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    return (
        X_train,
        X_test,
        y_train,
        y_test
    )

Now split SOH data:

In [9]:
X_train, X_test, y_train, y_test = group_split(
    X_soh,
    y_soh,
    soh_master["battery_id"]
)

**This code creates a complete machine-learning pipeline for predicting battery SOH**

SimpleImputer "median"

The imputer replaces each missing value with the median of its column

StandardScaler

Your features can have very different numerical ranges

StandardScaler transforms them to comparable scales



In [10]:
soh_model = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            max_depth=15,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    )
])

Train .....

In [11]:
soh_model.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

Pridict....

In [12]:
soh_predictions = soh_model.predict(
    X_test
)

Evaluate SOH model

In [13]:
soh_mae = mean_absolute_error(
    y_test,
    soh_predictions
)

soh_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        soh_predictions
    )
)

soh_r2 = r2_score(
    y_test,
    soh_predictions
)

print("SOH Model Results")

print("MAE :", soh_mae)
print("RMSE:", soh_rmse)
print("R2  :", soh_r2)

SOH Model Results
MAE : 0.0023497198867256787
RMSE: 0.0055194850292277576
R2  : 0.9979658804322377


MAE tells you the average prediction error.
Mae - on avg your prediction is off by this much if mae =0.5 your model avg error is 50000 simple and inclusive

rmse similar to mae but square the error firt this means large error are punished more severly than small error if ur model occasionaly pridicts 1 cr when the actual price is 50lakh rmse capture that catastrophic error much better than mae

r square ranges from 0 to 1 teels you waht percentage of price variation your model explain r sq is 0.65 means ur model explains 65 percent of why price varries the remaining 35 percent is noise missing features or randomness ur model cannot capture

in classification accuracy was ur main metric in regression you use ma,rmse r sq together because each tells u something different about ur model quality


**Train the RUL model**



In [14]:
RUL_FEATURES = [
    "cycle",
    "voltage",
    "temperature",
    "capacity",
    "capacity_loss",
    "capacity_retention",
    "voltage_drop_rate",
    "temperature_change_rate",
    "cycle_percentage"
]
X_rul = soh_master[RUL_FEATURES]

y_rul = soh_master["rul"]

SPLIT

In [15]:
X_train, X_test, y_train, y_test = group_split(
    X_rul,
    y_rul,
    soh_master["battery_id"]
)

Create model:

In [16]:
rul_model = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            max_depth=15,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
    )

])

Train

In [17]:
rul_model.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

Pridict

In [18]:
rul_predictions = rul_model.predict(
    X_test
)

In [19]:
print(
    "MAE:",
    mean_absolute_error(
        y_test,
        rul_predictions
    )
)

print(
    "RMSE:",
    np.sqrt(
        mean_squared_error(
            y_test,
            rul_predictions
        )
    )
)

print(
    "R2:",
    r2_score(
        y_test,
        rul_predictions
    )
)

MAE: 11.75669162947754
RMSE: 18.142002676765486
R2: 0.7242305219813129


Prepare the degradation dataset

In [20]:
degradation["Date_Time"] = pd.to_datetime(
    degradation["Date_Time"],
    errors="coerce"
)
degradation_master = (
    degradation
    .groupby("Cycle_Index")
    .agg({

        "Test_Time(s)": "max",

        "Current(A)": [
            "mean",
            "std",
            "min",
            "max"
        ],

        "Voltage(V)": [
            "mean",
            "std",
            "min",
            "max"
        ],

        "Charge_Capacity(Ah)": "max",

        "Discharge_Capacity(Ah)": "max",

        "Charge_Energy(Wh)": "max",

        "Discharge_Energy(Wh)": "max",

        "Internal_Resistance(Ohm)": [
            "mean",
            "max"
        ]

    })
    .reset_index()
)

In [21]:
degradation_master.columns = [

    "_".join(col).strip("_")

    if isinstance(col, tuple)

    else col

    for col in degradation_master.columns

]

CREATING TARGET

In [22]:
capacity = degradation_master[
    "Discharge_Capacity(Ah)_max"
].replace(0, np.nan)

In [23]:
degradation_master[
    "degradation_rate"
] = -capacity.pct_change()

In [24]:
degradation_master.dropna(
    subset=["degradation_rate"],
    inplace=True
)

Split degradation data chronologically

In [25]:
DEGRADATION_FEATURES = [

    column

    for column in degradation_master.select_dtypes(
        include=np.number
    ).columns

    if column != "degradation_rate"

]

Creating inputs

In [26]:
X = degradation_master[
    DEGRADATION_FEATURES
]

y = degradation_master[
    "degradation_rate"
]

SPLITING

In [27]:
split_index = int(
    len(X) * 0.8
)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

Create the thermal-risk target

In [28]:
thermal["temperature_rise"] = (
    thermal["Battery Temperature (End)"]
    -
    thermal[
        "Battery Temperature (Start) [°C]"
    ]
)

Create labels

In [29]:
conditions = [

    (
        thermal["Battery Temperature (End)"] >= 45
    )
    |
    (
        thermal["temperature_rise"] >= 12
    ),

    (
        thermal["Battery Temperature (End)"] >= 38
    )
    |
    (
        thermal["temperature_rise"] >= 7
    )

]

Assign classes

In [30]:
thermal["thermal_risk"] = np.select(
    conditions,
    [2, 1],
    default=0
)

In [31]:
print(
    thermal["thermal_risk"].value_counts()
)

#0 → Normal
#1 → Warning
#2 → Critical

thermal_risk
0    66
1     3
2     2
Name: count, dtype: int64


Selecting thermal features

In [32]:
THERMAL_FEATURES = [

    "Battery Temperature (Start) [°C]",

    "Battery State of Charge (Start)",

    "Ambient Temperature (Start) [°C]",

    "Target Cabin Temperature",

    "Distance [km]",

    "Duration [min]",

    "Average_Speed"

]

In [33]:
X_thermal = thermal[
    THERMAL_FEATURES
]

y_thermal = thermal[
    "thermal_risk"
]

In [34]:
thermal_model = Pipeline([

    (
        "imputer",
        SimpleImputer(strategy="median")
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42
        )
    )

])

Training the cycle-life model

FEATURE SELECTION

In [35]:
CYCLE_LIFE_FEATURES = [

    "IR",
    "QC",
    "QD",
    "Tavg",
    "Tmin",
    "Tmax",
    "chargetime",
    "cycle",
    "C1",
    "Q1",
    "C2",
    "Temperature_Range",
    "Normalized_IR",
    "Cycle_Percentage",
    "Charge_Discharge_Ratio"

]

In [36]:
X_cycle = cycle_life[
    CYCLE_LIFE_FEATURES
]
                                        #TARGET
y_cycle = cycle_life[
    "cycle_life"
]

 Group identifier

In [37]:
groups = cycle_life["battery_id"]

Create group-based train/test split

In [38]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_index, test_index = next(
    splitter.split(X_cycle, y_cycle, groups=groups)
)

 Create training and testing datasets

In [39]:
X_train = X_cycle.iloc[train_index]
X_test = X_cycle.iloc[test_index]

y_train = y_cycle.iloc[train_index]
y_test = y_cycle.iloc[test_index]

Train Random Forest model

In [40]:
cycle_life_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

cycle_life_model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

# Make predictions

In [41]:
predictions = cycle_life_model.predict(
    X_test
)


# Evaluate model
mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        predictions
    )
)

r2 = r2_score(
    y_test,
    predictions
)


print("Cycle Life Model Results")
print("------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

Cycle Life Model Results
------------------------
MAE : 123.7696191673829
RMSE: 212.36637088242688
R2  : 0.5653173985175657


In [42]:
train_batteries = set(
    cycle_life.iloc[train_index]["battery_id"]
)

test_batteries = set(
    cycle_life.iloc[test_index]["battery_id"]
)

common_batteries = (
    train_batteries.intersection(test_batteries)
)

print("Training batteries:")
print(train_batteries)

print("\nTesting batteries:")
print(test_batteries)

print("\nCommon batteries:")
print(common_batteries)

Training batteries:
{'b1c18', 'b1c29', 'b2c0', 'b2c5', 'b1c36', 'b2c16', 'b2c11', 'b1c14', 'b1c20', 'b3c45', 'b1c0', 'b1c10', 'b3c37', 'b1c17', 'b3c11', 'b3c24', 'b1c28', 'b3c44', 'b2c41', 'b1c21', 'b2c36', 'b1c8', 'b1c23', 'b2c30', 'b3c17', 'b1c13', 'b1c1', 'b2c9', 'b2c31', 'b1c7', 'b2c29', 'b1c27', 'b1c25', 'b1c30', 'b1c35', 'b3c7', 'b3c28', 'b1c39', 'b3c3', 'b2c4', 'b2c8', 'b1c38', 'b3c33', 'b2c39', 'b2c15', 'b1c44', 'b1c43', 'b1c5', 'b3c9', 'b1c3', 'b3c25', 'b3c12', 'b3c16', 'b2c20', 'b2c26', 'b2c2', 'b2c33', 'b2c17', 'b1c15', 'b2c22', 'b1c4', 'b2c47', 'b2c45', 'b2c19', 'b2c44', 'b2c13', 'b2c37', 'b2c35', 'b1c31', 'b1c16', 'b3c15', 'b1c9', 'b2c42', 'b2c24', 'b2c23', 'b1c34', 'b2c12', 'b3c14', 'b3c40', 'b3c1', 'b3c19', 'b3c2', 'b1c42', 'b3c13', 'b2c25', 'b1c24', 'b3c6', 'b2c1', 'b3c36', 'b3c32', 'b2c14', 'b3c34', 'b2c7', 'b2c10', 'b3c41', 'b3c20', 'b3c0', 'b1c40', 'b2c6', 'b3c35', 'b1c11', 'b2c32', 'b3c27', 'b2c46', 'b3c5', 'b2c43', 'b3c38', 'b3c29', 'b1c45', 'b2c34', 'b1c22', 'b3c4

#Train the battery-health model

In [43]:
HEALTH_FEATURES = [

    "type",

    "ambient_temperature",

    "Capacity",

    "Re",

    "Rct",

    "Resistance_Ratio"

]

In [44]:
print(
    health["type"].dtype
)

print(
    health["type"].unique()
)

int64
[-1  0  1]


In [45]:
health = pd.get_dummies(
    health,
    columns=["type"],
    drop_first=False
)

In [46]:
health_encoder = LabelEncoder()

health["health_encoded"] = (
    health_encoder.fit_transform(
        health["Health_Class"]
    )
)

In [47]:
# Define and train degradation_model
degradation_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])
degradation_model.fit(X_train, y_train)

# Define and train health_model
HEALTH_NEW_FEATURES = [col for col in health.columns if col.startswith('type_') or col in HEALTH_FEATURES and col != 'type']
X_health = health[HEALTH_NEW_FEATURES]
y_health = health['health_encoded']

health_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42
    ))
])
health_model.fit(X_health, y_health)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

#SAVING ALL MODELS

In [48]:
os.makedirs(
    "battery_apm_models",
    exist_ok=True
)

In [49]:
joblib.dump(
    soh_model,
    "battery_apm_models/soh_model.pkl"
)

joblib.dump(
    rul_model,
    "battery_apm_models/rul_model.pkl"
)

joblib.dump(
    degradation_model,
    "battery_apm_models/degradation_model.pkl"
)

joblib.dump(
    thermal_model,
    "battery_apm_models/thermal_model.pkl"
)

joblib.dump(
    cycle_life_model,
    "battery_apm_models/cycle_life_model.pkl"
)

joblib.dump(
    health_model,
    "battery_apm_models/health_model.pkl"
)

['battery_apm_models/health_model.pkl']

# Create the logical master table
Because the datasets describe different battery sources, the final master table should be created as a union, not an incorrect battery-ID merge.

In [50]:
soh_master["source_dataset"] = "SOH_RUL"

degradation_master[
    "source_dataset"
] = "DEGRADATION"

thermal[
    "source_dataset"
] = "THERMAL"

cycle_life[
    "source_dataset"
] = "CYCLE_LIFE"

health[
    "source_dataset"
] = "HEALTH"

In [51]:
master_training_table = pd.concat(
    [
        soh_master,
        degradation_master,
        thermal,
        cycle_life,
        health
    ],
    ignore_index=True,
    sort=False
)

In [52]:
master_training_table.to_csv(
    "master_battery_training_table.csv",
    index=False
)